In [7]:
import os
import glob
import torch
import torchaudio
import random

## Inflate Smaller Datasets (Target: 140)

In [8]:
def inflate_dataset(input_path, output_path, target_rows):
    """
    Oversamples a small dataset to reach a specific target row count.
    """
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = [line.strip() for line in f if line.strip()]

    current_rows = len(lines)
    print(f"📊 Current rows in {input_path}: {current_rows}")

    if current_rows == 0:
        print("❌ Error: Input file is empty.")
        return
    elif current_rows >= target_rows:
        print("✅ Dataset is already large enough. Copying as is.")
        final_lines = lines[:target_rows]
    else:
        # Calculate full duplications and the remaining gap
        full_copies = target_rows // current_rows
        remainder = target_rows % current_rows
        
        final_lines = lines * full_copies
        final_lines += random.sample(lines, remainder)

    # Shuffle to prevent the trainer from learning the duplication pattern
    random.shuffle(final_lines)

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in final_lines:
            f.write(line + '\n')
            
    print(f"🚀 Successfully inflated to {len(final_lines)} rows! Saved to {output_path}")

In [15]:
# ASD
inflate_dataset("/rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train.txt", "/rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train_inflated.txt", target_rows=140)

📊 Current rows in /rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train.txt: 73
🚀 Successfully inflated to 140 rows! Saved to /rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train_inflated.txt


In [11]:
# Aphasia
inflate_dataset("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train.txt", "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train_inflated.txt", target_rows=140)

📊 Current rows in /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train.txt: 95
🚀 Successfully inflated to 140 rows! Saved to /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train_inflated.txt


## Consolidating Train & Val Dataset

In [21]:
def combine_metadata(file_paths, output_path, seed=6033689):
    """
    Combines multiple metadata files, assigns strict Speaker IDs, and perfectly shuffles the output.
    """
    all_lines = []
    
    # Enumerate gives us an automatic 0, 1, 2 counter for our Speaker IDs
    for speaker_id, path in enumerate(file_paths):
        print(f"🛠️ Processing {path} -> Assigning Speaker ID: {speaker_id}")
        
        with open(path, 'r', encoding='utf-8') as in_f:
            for line in in_f:
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split('|')
                
                # Safety check to ensure it's a valid Matcha-TTS format row
                if len(parts) >= 3:
                    # Force the ID to match our new multi-speaker architecture
                    parts[1] = str(speaker_id)
                    new_line = '|'.join(parts)
                    all_lines.append(new_line)
                else:
                    print(f"⚠️ Skipping malformed line: {line}")
                    
    # 🔀 The Magic Step: Shuffle the entire combined dataset!
    print("🔀 Shuffling the combined dataset...")
    random.seed(seed) # Locks the randomness so you get the exact same shuffle if you re-run it
    random.shuffle(all_lines)
    
    # Write the beautifully mixed data to the final file
    with open(output_path, 'w', encoding='utf-8') as out_f:
        for line in all_lines:
            out_f.write(line + '\n')
            
    print(f"\n✅ Master metadata created at: {output_path} with {len(all_lines)} total rows.")

In [ ]:
# Create training file
files_to_combine = [
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train_inflated.txt",
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/dementia/train.txt",
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train_inflated.txt"
]

combine_metadata(files_to_combine, "/rds/general/user/ak8224/home/MedEmoji-TTS/data/master_train.txt")

🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/train_inflated.txt -> Assigning Speaker ID: 0
🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/dementia/train.txt -> Assigning Speaker ID: 1
🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/train_inflated.txt -> Assigning Speaker ID: 2
🔀 Shuffling the combined dataset...

✅ Master metadata created at: /rds/general/user/ak8224/home/MedEmoji-TTS/data/master_train.txt with 421 total rows.


In [24]:
# Create validation file
files_to_combine = [
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/val.txt",
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/dementia/val.txt",
    "/rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/val.txt"
]

combine_metadata(files_to_combine, "/rds/general/user/ak8224/home/MedEmoji-TTS/data/master_val.txt")

🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia/val.txt -> Assigning Speaker ID: 0
🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/dementia/val.txt -> Assigning Speaker ID: 1
🛠️ Processing /rds/general/user/ak8224/home/MedEmoji-TTS/data/asd/val.txt -> Assigning Speaker ID: 2
🔀 Shuffling the combined dataset...

✅ Master metadata created at: /rds/general/user/ak8224/home/MedEmoji-TTS/data/master_val.txt with 17 total rows.


## Calculating Global Mel-Spec Statistics

In [5]:
def get_combined_mel_stats(wav_dirs):
    print(f"🔍 Scanning {len(wav_dirs)} directories...")
    
    all_wav_files = []
    
    # 1. Loop through the list of directories and gather all file paths
    for wav_dir in wav_dirs:
        files = glob.glob(os.path.join(wav_dir, "*.wav"))
        all_wav_files.extend(files)
        print(f"  - Found {len(files)} files in {wav_dir}")
        
    if not all_wav_files:
        print("❌ No .wav files found! Check your directory paths.")
        return

    print(f"\n📊 Found {len(all_wav_files)} TOTAL files. Calculating Log-Mel Spectrograms...")

    # Match these EXACTLY to your yaml data parameters
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=22050,
        n_fft=1024,
        win_length=1024,
        hop_length=256,
        f_min=0.0,
        f_max=8000.0,
        n_mels=80,
        power=1.0, 
        normalized=False
    )

    all_mels = []
    
    for f in all_wav_files:
        waveform, sr = torchaudio.load(f)
        
        # Resample just in case any files aren't 22050Hz
        if sr != 22050:
            waveform = torchaudio.functional.resample(waveform, sr, 22050)
        
        # Convert to Mel
        mel = mel_transform(waveform)
        
        # Convert to Log-Mel 
        log_mel = torch.log(torch.clamp(mel, min=1e-5))
        all_mels.append(log_mel)

    print("\n🧠 Concatenating tensors...")
    # Stitch all audio frames together into one massive tensor
    all_mels_tensor = torch.cat(all_mels, dim=-1)
    
    # Calculate the global statistics
    mel_mean = all_mels_tensor.mean().item()
    mel_std = all_mels_tensor.std().item()

    print("\n✅ Calculation Complete!")
    print("-" * 40)
    print("data_statistics:")
    print(f"  mel_mean: {mel_mean:.6f}")
    print(f"  mel_std: {mel_std:.6f}")
    print("-" * 40)

In [6]:
directories = [
    "/rds/general/user/ak8224/home/emoji_project/data/Aphasia/final/preprocessed",
    "/rds/general/user/ak8224/home/emoji_project/data/Dementia/final/preprocessed",
    "/rds/general/user/ak8224/home/emoji_project/data/ASD/final/preprocessed"
]

get_combined_mel_stats(directories)

🔍 Scanning 3 directories...
  - Found 100 files in /rds/general/user/ak8224/home/emoji_project/data/Aphasia/final/preprocessed
  - Found 149 files in /rds/general/user/ak8224/home/emoji_project/data/Dementia/final/preprocessed
  - Found 77 files in /rds/general/user/ak8224/home/emoji_project/data/ASD/final/preprocessed

📊 Found 326 TOTAL files. Calculating Log-Mel Spectrograms...

🧠 Concatenating tensors...

✅ Calculation Complete!
----------------------------------------
data_statistics:
  mel_mean: -0.842103
  mel_std: 1.573202
----------------------------------------
